# Model management system database access

Anton Antonov   
June 2026   
July 2026

---

## Introduction

This notebook shows how to connect to the Model Management System (MMS) database of the Geo-Spatial Pricing Engine (GSPE).

---

## Setup

Packages for data frame manipulation and import:

In [1]:
import pandas as pd
import json

Using the (newer, recommended) modern `psycopg` (`psycopg3`):

In [2]:
import psycopg

In [3]:
DB_CONFIG = {
    'dbname': 'geo_spatial_pricing_engine',   # Change this
    'user': 'postgres',               # or your username
    'password': '',
    'host': 'localhost',              # or IP / remote host
    'port': '5432'
}

Connect to the database with:

```python
with psycopg.connect(**DB_CONFIG) as conn:
    ...
```

or similar.

Load the chatbook extension:

In [4]:
%load_ext JupyterChatbook

---

## Table information

Query to get all tables (excluding system tables):

In [4]:
with psycopg.connect(**DB_CONFIG) as conn:
    with conn.cursor() as cur:
        cur.execute("""
            SELECT table_schema, table_name, table_type
            FROM information_schema.tables
            WHERE table_schema NOT IN ('information_schema', 'pg_catalog')
            ORDER BY table_schema, table_name;
        """)
        
        tables = cur.fetchall()
        
        print(f"\n📋 Tables in '{DB_CONFIG['dbname']}':\n")
        for schema, table, ttype in tables:
            print(f"{schema:.<15} {table}")


📋 Tables in 'geo_spatial_pricing_engine':

public......... calibrated_value
public......... experiment
public......... experimental_result
public......... geo_taxonomy
public......... model
public......... model_parameter
public......... raw_transportation_trips
public......... tile_data
public......... transportation_trips


----

## Raw transportation trips data

In [6]:
fileName = "../../resources/RawTransportationTrips.json"
dfRawTrips = pd.DataFrame(json.load(open(file=fileName)))
dfRawTrips.shape

(4556, 11)

In [7]:
dfRawTrips[1003:1006]

,start_city,start_state,start_zip_code,start_lat,start_lon,end_city,end_state,end_zip_code,end_lat,end_lon,distance
1003,Columbus,Ohio,None,39.988475,-82.988793,Washington,DistrictOfColumbia,None,38.904148,-77.017094,411.847423
1004,Columbus,Ohio,None,39.988475,-82.988793,Wichita,Kansas,None,37.689363,-97.343805,852.893641
1005,Dallas,Texas,None,32.794176,-96.765503,Albany,NewYork,None,42.665745,-73.798353,1640.481347


In [ ]:
if False:
    with psycopg.connect(**DB_CONFIG) as conn:
        with conn.cursor() as cur:
            query = """
                INSERT INTO raw_transportation_trips (
                    start_city, start_state, start_zip_code, start_lat, start_lon,
                    end_city, end_state, end_zip_code, end_lat, end_lon, distance,
                    raw_transportation_trips_id, price
                )
                VALUES (
                    %(start_city)s, %(start_state)s, %(start_zip_code)s,
                    %(start_lat)s, %(start_lon)s,
                    %(end_city)s, %(end_state)s, %(end_zip_code)s,
                    %(end_lat)s, %(end_lon)s, %(distance)s,
                    'FAFDerived', 0
                )
            """
            cur.executemany(query, dfRawTrips.to_dict("records"))
        conn.commit()

---

## Geo-taxonomy

In [8]:
fileName = "../../resources/Hextile1deg.json"

dfTaxonomy = pd.DataFrame(json.load(open(fileName))).astype({
    "Taxonomy": str,
    "Tag": str,
    "CenterLat": float,
    "CenterLon": float,
    "Coordinates": str
})

dfTaxonomy.shape

(1043, 5)

In [9]:
dfTaxonomy[100:105]

,Taxonomy,Tag,CenterLon,CenterLat,Coordinates
100,Hextile1deg,tile00101,-118.0,45.033321,"[[-118.5, 45.32199613138562], [-118.5, 44.7446..."
101,Hextile1deg,tile00102,-118.0,46.765372,"[[-118.5, 47.05404693895449], [-118.5, 46.4766..."
102,Hextile1deg,tile00103,-118.0,48.497423,"[[-118.5, 48.786097746523374], [-118.5, 48.208..."
103,Hextile1deg,tile00104,-117.5,33.774991,"[[-118.0, 34.06366588218792], [-118.0, 33.4863..."
104,Hextile1deg,tile00105,-117.5,35.507042,"[[-118.0, 35.795716689756794], [-118.0, 35.218..."


In [10]:
if False:
    with psycopg.connect(**DB_CONFIG) as conn:
        with conn.cursor() as cur:
            query = """
                INSERT INTO geo_taxonomy (
                    geo_taxonomy_id, tile_id, center_lat, center_lon, coordinates)
                VALUES (
                    %(Taxonomy)s, %(Tag)s, %(CenterLat)s, %(CenterLon)s, %(Coordinates)s::jsonb
                )
            """
            cur.executemany(query, dfTaxonomy.to_dict("records"))
        conn.commit()

----

## Geo-taxonomy tile data

In [11]:
fileName = "../../resources/Hextile1degTileData.json"
dfTileData = pd.DataFrame(json.load(open(fileName)))
dfTileData.shape

(1043, 4)

In [12]:
dfTileData[200:204]

,Taxonomy,Tag,Elevation,Population
200,Hextile1deg,tile00201,5613.517060,2923
201,Hextile1deg,tile00202,5849.737533,4367
202,Hextile1deg,tile00203,5990.813648,4584
203,Hextile1deg,tile00204,4704.724409,6423


In [13]:
dfTileDataLong = dfTileData.copy()
dfTileDataLong = pd.melt(dfTileDataLong, id_vars = dfTileDataLong.columns[0:2], var_name = "Variable", value_name = "Value" )

In [14]:
dfTileDataLong[200:202]

,Taxonomy,Tag,Variable,Value
200,Hextile1deg,tile00201,Elevation,5613.517060
201,Hextile1deg,tile00202,Elevation,5849.737533


In [15]:
if False:
    with psycopg.connect(**DB_CONFIG) as conn:
        with conn.cursor() as cur:
            query = """
                INSERT INTO tile_data (
                    geo_taxonomy_id, tile_data_id, tile_id, name, value)
                VALUES (
                    %(Taxonomy)s, 'Hextile1degTileData', %(Tag)s, %(Variable)s, %(Value)s
                )
            """
            cur.executemany(query, dfTileDataLong.to_dict("records"))
        conn.commit()